# 03 | ANOVA Analysis
**Final Individual Project | YRBS 2007**

## Research Question

> **Is there a significant difference in mean height among students with different levels of milk drinking frequency?**

### Analysis Variables

| Role | Variable | Definition |
|---|---|---|
| Explanatory | `NoMilkDrinking` | Milk drinking frequency, recoded into three ordered groups |
| Response | `HowTallAreYouWithoutShoesInMeters` | Self-reported height without shoes, measured in meters |
### alpha = 0.05

## Hypothesis Test

**H0 : μ1 = μ2 = μ3 (All group means are equal)**  
**H1 : At least one group mean is different**  

Where:  
μ1 = No milk drinking (0)  
μ2 = Occasional milk drinking (1)  
μ3 = Frequent milk drinking (2)

In [9]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols
from IPython.display import display, HTML

# Load dataset
df = pd.read_csv("../data/processed/YRBS_Cleaned_and_Recoded.csv")
df["NoMilkDrinking"] = df["NoMilkDrinking"].astype("category")

In [11]:
# Fit ANOVA model
model = ols(
    "HowTallAreYouWithoutShoesInMeters ~ NoMilkDrinking",
    data=df
).fit()

anova_table = sm.stats.anova_lm(model, typ=2)

# Copy result table
anova_core = anova_table.copy()

# Rename columns
anova_core = anova_core.rename(
    columns={
        "sum_sq": "SS",
        "df": "df",
        "F": "F-value",
        "PR(>F)": "p-value"
    }
)

# Compute Mean Square
anova_core["MS"] = anova_core["SS"] / anova_core["df"]

# Rename row labels
anova_core = anova_core.rename(
    index={
        "NoMilkDrinking": "Milk Drinking Group (Between Groups)",
        "Residual": "Residual (Within Groups)"
    }
)

# Format p-values
def format_p(p):
    if pd.isna(p):
        return "-"
    if p < 0.001:
        return "<0.001"
    return round(p, 3)

anova_core["p-value"] = anova_core["p-value"].apply(format_p)

anova_core = anova_core.fillna("-")

# Reorder columns
anova_core = anova_core[
    ["SS", "df", "MS", "F-value", "p-value"]
]

# Add Total row
total_row = pd.DataFrame({
    "SS": [anova_core["SS"].sum()],
    "df": [anova_core["df"].sum()],
    "MS": [anova_core["SS"].sum() / anova_core["df"].sum()],
    "F-value": ["-"],
    "p-value": ["-"]
}, index=["Total"])

anova_core = pd.concat([anova_core, total_row])

# Display title
display(HTML("""
<h4 style="
    margin-left:270px;
    font-weight:bold;
    margin-bottom:10px;">
    ANOVA Table
</h4>
"""))

# Format table
styled_anova = (
    anova_core.style
    .set_properties(**{"text-align": "left"})
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "left")]}
    ])
)

display(styled_anova)

,SS,df,MS,F-value,p-value
Milk Drinking Group (Between Groups),2.801031,2.000000,1.400516,139.055874,<0.001
Residual (Within Groups),127.556868,12665.000000,0.010072,-,-
Total,130.357900,12667.000000,0.010291,-,-


In [33]:
from IPython.display import display, HTML

# Get p-value and F-value
p_value = anova_table["PR(>F)"].iloc[0]
f_value = anova_table["F"].iloc[0]

display(HTML("""
<h5 style="
    margin-left:75px;
    font-weight:bold;
    margin-bottom:10px;">
    ANOVA Test Result
</h5>
"""))

# Print test statistics
print(f"F-value: {f_value:.3f}")
print(f"p-value: {p_value:.6e}")

# Interpret statistical significance
if p_value < 0.001:
    print("Result: Highly significant (p < 0.001)")
elif p_value < 0.05:
    print("Result: Significant (p < 0.05)")
else:
    print("Result: Not significant (p ≥ 0.05)")

# Get sum of squares
ss_between = anova_table["sum_sq"].iloc[0]
ss_total = anova_table["sum_sq"].sum()

# Compute eta squared (effect size)
eta_sq = ss_between / ss_total

display(HTML("""
<h5 style="
    margin-left:65px;
    font-weight:bold;
    margin-top:20px;
    margin-bottom:10px;">
    Effect Size
</h5>
"""))

# Print effect size
print(f"Eta Squared (η²): {eta_sq:.6f}")

# Interpret effect size
if eta_sq < 0.01:
    effect = "Very small"
elif eta_sq < 0.06:
    effect = "Small"
elif eta_sq < 0.14:
    effect = "Medium"
else:
    effect = "Large"

print(f"Interpretation: {effect} effect")

F-value: 139.056
p-value: 1.829444e-60
Result: Highly significant (p < 0.001)


Eta Squared (η²): 0.021487
Interpretation: Small effect


## Interpretation
Although the ANOVA test showed **a statistically significant difference among groups (p < 0.001)**, the **effect size was small (η² ≈ 0.02)**, indicating that the practical difference in height across milk consumption groups is limited.